## Preprocessing

notebook ini menyiapkan data untuk modeling. hasilnya disimpan di `data/processed/`, file asli di `data/` tidak diubah.

yang dikerjakan di sini hanya cleaning yang **tidak belajar dari data** (aturan tetap, berlaku per baris). imputasi dan scaling belajar dari data (median, mean, std), jadi dikerjakan di notebook modeling di dalam pipeline, di-fit di fold train saja supaya tidak bocor ke validasi/test.

urutan:
1. load data
2. seleksi subjek (tanpa riwayat DM)
3. perbaiki nilai (recode, kode 9, outlier → NaN)
4. buang kolom leakage dan kolom yg tidak dipakai
5. buang kolom redundan, kosong > 50%, dan hampir konstan
6. bentuk target dan feature set (F1 non-lab, F2 combined)
7. cek ulang leakage di fitur akhir
8. split 80:20 (test dikunci)
9. simpan

### Import Dependencies

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)

RANDOM_STATE = 42
TEST_SIZE = 0.20

DATA_MENTAH = Path("data/Dataset for diabetes research.csv")
FOLDER_OUT = Path("data/processed")
FOLDER_OUT.mkdir(parents=True, exist_ok=True)

NAMA_KELAS = {0: "NGR", 1: "Prediabetes", 2: "DM"}

# semua perubahan dicatat di sini, nanti disimpan jadi preprocessing_log.csv
LOG = []
def catat(tahap, kolom, aksi, n):
    LOG.append({"tahap": tahap, "kolom": kolom, "aksi": aksi, "n_terdampak": int(n)})

### 1. Load Data

sama seperti di EDA: sel berisi spasi jadi NaN, lalu jadi angka kecuali 3 kolom teks.

In [2]:
KOLOM_TEKS = ["Data", "EscDrinkD", "EyeDia"]

df = pd.read_csv(DATA_MENTAH, encoding="gbk", dtype=str)
df = df.replace(r"^\s*$", np.nan, regex=True)
kolom_angka = df.columns.drop(KOLOM_TEKS)
df[kolom_angka] = df[kolom_angka].apply(pd.to_numeric, errors="coerce")

print("data mentah:", df.shape)

data mentah: (5922, 190)


/sessions/rcw-01gensfq8hlmz6f9bgnhmefx/tmp/ipykernel_11/4280057060.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(r"^\s*$", np.nan, regex=True)


### 2. Seleksi Subjek

tujuan: skrining orang yg **belum tahu** dirinya prediabetes/DM. orang dengan riwayat DM (`DMhis = 1`) dikeluarkan.

In [3]:
n_awal = len(df)
df = df[df["DMhis"] == 0].copy().reset_index(drop=True)
catat("seleksi subjek", "DMhis", "buang baris DMhis = 1", n_awal - len(df))

print("sisa:", len(df), "orang")
df["PGclass3"].map(NAMA_KELAS).value_counts()

sisa: 5873 orang


PGclass3
NGR            4444
Prediabetes    1121
DM              308
Name: count, dtype: int64

### 3. Perbaiki Nilai

#### 3a. kosong yg artinya "tidak"

beberapa kolom cuma diisi 1 kalau jawabannya ya (sudah dibahas di EDA), jadi kosong = 0.

In [4]:
for kolom in ["HypertenHis", "DyslipideHis"]:
    n = df[kolom].isna().sum()
    df[kolom] = df[kolom].fillna(0)
    catat("perbaiki nilai", kolom, "kosong -> 0 (artinya tidak)", n)

#### 3b. gender

kamus data bilang 0/1, tapi data isinya 1/2. dari EDA: 1 = laki-laki, 2 = perempuan. dibuat kolom baru `Male` (1 = laki-laki, 0 = perempuan).

In [5]:
df["Male"] = (df["Gender"] == 1).astype(int)
catat("perbaiki nilai", "Gender", "recode jadi Male (1 = laki-laki)", len(df))
pd.crosstab(df["Gender"], df["Male"])

Male,0,1
Gender,,
1,0,2741
2,3132,0


#### 3c. kode 9 = tidak tahu

di kolom riwayat penyakit, `WeightLoss`, `FatherDM`, `MotherDM`, angka 9 artinya "tidak tahu". kalau dibiarkan, model menganggap 9 lebih besar dari 1, padahal bukan. jadi 9 → NaN.

In [6]:
RIWAYAT = ["Angina", "Arrhythmia", "PAD", "Hyper16", "Hyper13", "Blind", "CerebralHe", "CerebralIn",
           "TIA", "MI", "HF", "CAP", "Hypo14", "RF", "Amputation"]
KODE_9 = RIWAYAT + ["WeightLoss", "FatherDM", "MotherDM"]

for kolom in KODE_9:
    n = (df[kolom] == 9).sum()
    df.loc[df[kolom] == 9, kolom] = np.nan
    catat("perbaiki nilai", kolom, "9 (tidak tahu) -> NaN", n)

pd.DataFrame(LOG).query("aksi.str.startswith('9')", engine="python")

,tahap,kolom,aksi,n_terdampak
4,perbaiki nilai,Angina,9 (tidak tahu) -> NaN,972
5,perbaiki nilai,Arrhythmia,9 (tidak tahu) -> NaN,1118
6,perbaiki nilai,PAD,9 (tidak tahu) -> NaN,1122
7,perbaiki nilai,Hyper16,9 (tidak tahu) -> NaN,1116
8,perbaiki nilai,Hyper13,9 (tidak tahu) -> NaN,1111
9,perbaiki nilai,Blind,9 (tidak tahu) -> NaN,943
10,perbaiki nilai,CerebralHe,9 (tidak tahu) -> NaN,903
11,perbaiki nilai,CerebralIn,9 (tidak tahu) -> NaN,904
12,perbaiki nilai,TIA,9 (tidak tahu) -> NaN,1008
13,perbaiki nilai,MI,9 (tidak tahu) -> NaN,916


#### 3d. outlier yg tidak masuk akal

- kreatinin (`SCRE`) < 35 µmol/L terlalu rendah untuk orang dewasa → NaN. karena eGFR (`GFR`) dihitung dari kreatinin, eGFR baris tsb juga → NaN.
- tinggi < 120 cm → NaN (tinggi, dan BMI ikut dihitung dari tinggi).
- tensi atas ≤ tensi bawah → dua-duanya NaN.
- LDL > kolesterol total → LDL NaN.

In [7]:
kreatinin_aneh = df["SCRE"] < 35
df.loc[kreatinin_aneh, ["SCRE", "GFR"]] = np.nan
catat("outlier", "SCRE, GFR", "SCRE < 35 -> NaN", kreatinin_aneh.sum())

tinggi_aneh = df["Height"] < 120
df.loc[tinggi_aneh, ["Height", "BMI"]] = np.nan
catat("outlier", "Height, BMI", "Height < 120 -> NaN", tinggi_aneh.sum())

tensi_aneh = df["Bpsys"] <= df["Bpdia"]
df.loc[tensi_aneh, ["Bpsys", "Bpdia"]] = np.nan
catat("outlier", "Bpsys, Bpdia", "Bpsys <= Bpdia -> NaN", tensi_aneh.sum())

ldl_aneh = df["LDL"] > df["CHOL"]
df.loc[ldl_aneh, "LDL"] = np.nan
catat("outlier", "LDL", "LDL > CHOL -> NaN", ldl_aneh.sum())

print("eGFR > 200 yg masih tersisa:", (df["GFR"] > 200).sum())
pd.DataFrame(LOG).query("tahap == 'outlier'")

eGFR > 200 yg masih tersisa: 84


,tahap,kolom,aksi,n_terdampak
22,outlier,"SCRE, GFR",SCRE < 35 -> NaN,327
23,outlier,"Height, BMI",Height < 120 -> NaN,2
24,outlier,"Bpsys, Bpdia",Bpsys <= Bpdia -> NaN,2
25,outlier,LDL,LDL > CHOL -> NaN,3


### 4. Buang Kolom Leakage & Tidak Dipakai

daftar leakage sama dengan EDA. tambahan baru:

- `FINS` dan `FCP` (insulin dan C-peptide puasa). bukan glukosa, tapi penanda metabolisme glukosa dan bukan lab rutin. kalau dipakai, klaim "tanpa tes gula" jadi lemah.
- `DMhighrisk` (kelompok risiko tinggi DM). definisinya tidak dijelaskan di kamus data. kriteria risiko tinggi di pedoman China biasanya memasukkan riwayat gangguan toleransi glukosa, jadi bisa bocor sebagian. dibuang supaya aman.

In [8]:
BOCOR = [
    "FPG", "PG2h", "GLU", "DMhis",
    "HomaIR", "Homaβ", "ISIGutt", "INS2h", "CP2h",
    "FPGover7", "P2PGover11",
    "DM", "DMnew3class", "PGclass6",
    "VAR00001", "ThalussemiaAND",
]
TIDAK_DIPAKAI = [
    "HbA1c", "HPLC", "GA", "A1cover6575",
    "filter", "GAstudy", "GAnum",
    "Known", "OPD", "ToChilren", "DietExcise", "Compl", "Check", "GlucoseLev",
    "DF", "MADPN", "TFDPN", "DR", "DRyd",
    "CDSMS",
]
TAMBAHAN = ["FINS", "FCP", "DMhighrisk"]
ID_TEKS = ["NO", "Gender", "ALT.1"] + KOLOM_TEKS     # Gender diganti Male, ALT.1 = akar ALT

for kolom in BOCOR:
    catat("leakage", kolom, "buang (leakage)", 0)
for kolom in TIDAK_DIPAKAI:
    catat("tidak dipakai", kolom, "buang (glikemik lain / penanda studi / kuesioner)", 0)
for kolom in TAMBAHAN:
    catat("tidak dipakai", kolom, "buang (penanda metabolik glukosa / definisi tidak jelas)", 0)

kolom_buang = set(BOCOR + TIDAK_DIPAKAI + TAMBAHAN + ID_TEKS)
kandidat = [c for c in df.columns if c not in kolom_buang and c != "PGclass3"]
print("kandidat fitur:", len(kandidat))

kandidat fitur: 145


### 5. Buang Kolom Redundan, Kosong, Konstan

#### 5a. redundan

dari EDA: banyak kolom cuma potongan (cut-off) dari kolom angka, atau hasil hitungan dari kolom lain. yg dipertahankan versi angka aslinya.

In [9]:
REDUNDAN = {
    "Age3Group": "potongan Age", "AgeGroup": "potongan Age",
    "BMI4Class": "potongan BMI", "overweight": "potongan BMI", "obesity": "potongan BMI",
    "WGOCobBMI": "potongan BMI", "BMI25": "potongan BMI",
    "WC9085": "potongan waist1", "WC8580": "potongan waist1", "WC9080": "potongan waist1",
    "Waist2": "lingkar pinggang versi lain (umbilikal)",
    "BP13085": "potongan Bpsys/Bpdia", "WHO1999hbp": "potongan Bpsys/Bpdia",
    "Fat2Class": "potongan Fat", "Fat3Class": "potongan Fat",
    "ALTn": "potongan ALT", "TBILn": "potongan TBIL", "FIB4n": "potongan FIB4",
    "IDBIL": "TBIL - DBIL", "GLO": "TP - ALB", "AG": "ALB / GLO",
    "TGdis": "potongan TG", "CDSdyelip": "potongan lipid",
    "DysHDLIDFNCEP": "potongan HDL", "JCDHDLdis": "potongan HDL",
    "highCr": "potongan SCRE", "HighUA": "potongan SUA",
    "HighUmALB": "potongan UmALB", "DN": "potongan UmALB", "ACR3degree": "potongan UMAUCR",
    "HighACR": "potongan UMAUCR", "UACRGFR": "potongan UMAUCR & GFR",
    "EUGFRabACR012": "potongan GFR & UMAUCR", "EUGFR90UACR01": "potongan GFR & UMAUCR",
    "EUGFR60abACR012": "potongan GFR & UMAUCR",
    "GFR5lev": "potongan GFR", "GFR6levG5": "potongan GFR",
    "Hb4per": "potongan HGB", "Anemia": "potongan HGB", "HCT": "r = 0,90 dengan HGB",
    "LCcount": "versi count dari LC", "EOcount": "versi count dari EO", "ALYcount": "versi count dari ALY",
    "Smoking2": "potongan Smoking", "Drinking2": "potongan Drinking",
    "FatherDM": "sudah dirangkum DMfamilyHistory", "MotherDM": "sudah dirangkum DMfamilyHistory",
    "BSDM": "sudah dirangkum DMfamilyHistory", "ChildDM": "sudah dirangkum DMfamilyHistory",
}
for kolom, alasan in REDUNDAN.items():
    if kolom in kandidat:
        catat("redundan", kolom, f"buang ({alasan})", 0)

kandidat = [c for c in kandidat if c not in REDUNDAN]
print("sisa setelah buang redundan:", len(kandidat))

sisa setelah buang redundan: 96


#### 5b. kosong > 50%, konstan, dan hampir konstan

kolom riwayat penyakit seperti `MI` cuma punya 2 orang bernilai 1. kolom seperti ini tidak bisa dipelajari model dan membuat hasil tidak stabil antar fold. aturannya: kolom biner dengan kelas minoritas < 20 orang dibuang.

In [10]:
MIN_MINORITAS = 20

def alasan_buang(s):
    if s.isna().mean() > 0.5:
        return "kosong > 50%"
    if s.nunique() <= 1:
        return "konstan"
    vc = s.value_counts()
    if s.nunique() == 2 and vc.min() < MIN_MINORITAS:
        return f"hampir konstan (minoritas {vc.min()} orang)"
    return None

buang_5b = {c: alasan_buang(df[c]) for c in kandidat}
buang_5b = {c: a for c, a in buang_5b.items() if a}
for kolom, alasan in buang_5b.items():
    catat("seleksi kolom", kolom, f"buang ({alasan})", 0)

fitur = [c for c in kandidat if c not in buang_5b]
print("dibuang:", len(buang_5b))
print("fitur akhir:", len(fitur))
pd.Series(buang_5b, name="alasan").to_frame()

dibuang: 25
fitur akhir: 71


,alasan
HCVAb,hampir konstan (minoritas 9 orang)
PLCR,kosong > 50%
ThalussemiaHPLC,kosong > 50%
SmokingAge,kosong > 50%
SmokingRange,kosong > 50%
DrinkingAge,kosong > 50%
Drink50g,kosong > 50%
EscDrink,kosong > 50%
EscDrinkM,kosong > 50%
EscDrinkT,kosong > 50%


### 6. Target dan Feature Set

target:
- `y3`: 3 kelas (0 = NGR, 1 = prediabetes, 2 = DM)
- `y_dm`: DM vs bukan DM (skrining DM yg belum terdiagnosis)
- `y_abn`: abnormal (prediabetes + DM) vs NGR

feature set (RQ1):
- **F1 non-lab**: yg bisa didapat tanpa ambil darah: demografi, antropometri, tensi, gaya hidup, riwayat.
- **F2 combined**: F1 + lab rutin non-glikemik (hati, lipid, ginjal, darah lengkap, urin).

In [11]:
df["y3"] = df["PGclass3"].astype(int)
df["y_dm"] = (df["y3"] == 2).astype(int)
df["y_abn"] = (df["y3"] >= 1).astype(int)

NON_LAB_SEMUA = (
    ["Male", "Age", "Height", "Weight", "BMI", "waist1", "hip", "WHR", "leg", "Fat", "HR", "Bpsys", "Bpdia",
     "HypertenHis", "DyslipideHis", "Smoking", "Drinking", "Tea", "WeightLoss", "DMfamilyHistory", "cataract"]
    + RIWAYAT
)
F1 = [c for c in fitur if c in NON_LAB_SEMUA]
F2 = fitur.copy()
LAB = [c for c in F2 if c not in F1]

print(f"F1 non-lab ({len(F1)}):", F1)
print(f"\nlab rutin ({len(LAB)}):", LAB)
print(f"\nF2 combined: {len(F2)} fitur")

F1 non-lab (25): ['Age', 'Height', 'Weight', 'BMI', 'waist1', 'hip', 'WHR', 'leg', 'HR', 'Bpsys', 'Bpdia', 'Fat', 'HypertenHis', 'DyslipideHis', 'Smoking', 'Drinking', 'Tea', 'Angina', 'Arrhythmia', 'Hyper13', 'Blind', 'WeightLoss', 'DMfamilyHistory', 'cataract', 'Male']

lab rutin (46): ['ALT', 'AST', 'ASTALT', 'GGT', 'ALP', 'TBIL', 'DBIL', 'TP', 'ALB', 'HbsAg', 'CHOL', 'TG', 'HDL', 'LDL', 'BUN', 'SCRE', 'SUA', 'UCRE', 'UmALB', 'UMAUCR', 'WBC', 'LC', 'ALY', 'RBC', 'RDWCV', 'RDWSD', 'MCV', 'LYMPH', 'LYMPHcount', 'MCH', 'MCHC', 'MPV', 'BASO', 'BASOcount', 'EO', 'HGB', 'PLT', 'PDW', 'PCT', 'MONO', 'MONOcount', 'NEUT', 'NEUTcount', 'FIB4', 'ThalussemiaOR', 'GFR']

F2 combined: 71 fitur


### 7. Cek Ulang Leakage di Fitur Akhir

setiap fitur diuji sendirian: seberapa bagus satu fitur saja memisahkan abnormal vs NGR (AUC). kalau ada fitur yg AUC-nya sangat tinggi (> 0,75), curigai bocor. diuji juga pola kosongnya: kalau kolom hanya diisi untuk kelas tertentu, kosongnya sendiri bisa jadi bocoran.

catatan: ini dihitung di seluruh data hanya untuk audit, tidak dipakai untuk memilih fitur.

In [12]:
def auc_satu_fitur(x, y):
    try:
        a = roc_auc_score(y, x)
        return max(a, 1 - a)
    except ValueError:
        return np.nan

audit = []
for kolom in F2:
    x = df[kolom]
    audit.append({
        "fitur": kolom,
        "feature_set": "F1" if kolom in F1 else "lab",
        "auc_nilai": auc_satu_fitur(x.fillna(x.median()), df["y_abn"]),
        "auc_kosong": auc_satu_fitur(x.isna().astype(int), df["y_abn"]),
        **{f"kosong_{NAMA_KELAS[k]}_%": x[df["y3"] == k].isna().mean() * 100 for k in NAMA_KELAS},
    })
audit = pd.DataFrame(audit).sort_values("auc_nilai", ascending=False).round(3)

print("fitur dengan auc_nilai > 0,75 :", (audit["auc_nilai"] > 0.75).sum())
print("fitur dengan auc_kosong > 0,6 :", (audit["auc_kosong"] > 0.6).sum())
audit.head(15)

fitur dengan auc_nilai > 0,75 : 0
fitur dengan auc_kosong > 0,6 : 0


,fitur,feature_set,auc_nilai,auc_kosong,kosong_NGR_%,kosong_Prediabetes_%,kosong_DM_%
15,GGT,lab,0.648,0.502,0.923,0.624,0.325
0,Age,F1,0.637,0.500,0.000,0.000,0.000
9,Bpsys,F1,0.621,0.505,2.183,1.160,1.623
10,Bpdia,F1,0.605,0.505,2.183,1.160,1.623
4,waist1,F1,0.604,0.506,1.958,0.803,0.974
6,WHR,F1,0.603,0.506,1.958,0.803,0.974
55,FIB4,lab,0.603,0.502,1.170,0.892,0.000
13,AST,lab,0.598,0.501,0.338,0.089,0.000
53,NEUT,lab,0.590,0.501,1.215,1.249,0.325
12,ALT,lab,0.588,0.501,0.360,0.089,0.000


### 8. Split 80:20

stratified berdasarkan `y3`. test set **dikunci**: tidak dipakai sama sekali sampai model final dipilih. semua eksperimen CV di notebook modeling hanya pakai bagian train.

In [13]:
idx_train, idx_test = train_test_split(
    df.index, test_size=TEST_SIZE, stratify=df["y3"], random_state=RANDOM_STATE
)
df["split"] = "train"
df.loc[idx_test, "split"] = "test"

pd.crosstab(df["y3"].map(NAMA_KELAS), df["split"], margins=True)

split,test,train,All
y3,,,
DM,62,246,308
NGR,889,3555,4444
Prediabetes,224,897,1121
All,1175,4698,5873


### 9. Simpan

In [14]:
kolom_simpan = ["NO", "split", "y3", "y_dm", "y_abn"] + F2
df[kolom_simpan].to_csv(FOLDER_OUT / "diabetes_clean.csv", index=False)

feature_sets = {
    "F1_nonlab": F1,
    "F2_combined": F2,
    "lab_only": LAB,
    "cdrs": ["Age", "Male", "BMI", "waist1", "Bpsys", "DMfamilyHistory"],
}
meta = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "n_total": int(len(df)),
    "n_train": int((df["split"] == "train").sum()),
    "n_test": int((df["split"] == "test").sum()),
    "target": {"y3": NAMA_KELAS, "y_dm": "1 = DM", "y_abn": "1 = prediabetes atau DM"},
    "feature_sets": feature_sets,
}
with open(FOLDER_OUT / "feature_sets.json", "w") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

pd.DataFrame(LOG).to_csv(FOLDER_OUT / "preprocessing_log.csv", index=False)
audit.to_csv(FOLDER_OUT / "leakage_audit.csv", index=False)

for p in sorted(FOLDER_OUT.iterdir()):
    print(p)

data/processed/diabetes_clean.csv
data/processed/feature_sets.json
data/processed/leakage_audit.csv
data/processed/preprocessing_log.csv


### Ringkasan

- subjek: orang tanpa riwayat DM, dibagi train 80% dan test 20% (test dikunci).
- nilai diperbaiki: gender di-recode, kode 9 → NaN, outlier kreatinin/eGFR, tinggi, tensi, dan LDL → NaN.
- kolom leakage, glikemik lain, insulin/C-peptide puasa, dan `DMhighrisk` dibuang.
- kolom redundan, kosong > 50%, dan hampir konstan dibuang. log lengkap ada di `preprocessing_log.csv`.
- **NaN tidak diisi di sini.** imputasi dan scaling dilakukan di notebook modeling di dalam pipeline (fit di train fold saja).